In [1]:
import pandas as pd

df = pd.read_csv("wb_2026_results.csv")
parties = sorted(
    c.removesuffix("_total_votes")
    for c in df.columns
    if c.endswith("_total_votes")
)

for p in parties:
    print(p)
print(f"\n{len(parties)} unique parties")

Aam Janata Unnayan party
Akhil Bharatiya Gorkha League
All India Forward Bloc
All India Majlis-E-Inquilab-E-Millat
All India Majlis-E-Ittehadul Muslimeen
All India Secular Front
All India Trinamool Congress
Ambedkarite Party of India
Amra Bangalee
Bahujan Mukti Party
Bahujan Samaj Party
Bharateeya Manavadhikar party
Bharatiya Gorkha Prajatantrik Morcha
Bharatiya Janata Party
Bharatiya National Janata Dal
Bharatiya Nyay-Adhikar Raksha Party
Bharatiya Rashtriya Forward Bloc
Bhartiya Lokmat Rashtrawadi Party
Bhartiya Rashtriya Dal
Bhumiputra United Party
Communist Party of India
Communist Party of India (Marxist)
Communist Party of India (Marxist-Leninist) (Liberation)
Gana Suraksha Party
Gondvana Gantantra Party
Guruchand Sena Dal
Hindu Samaj Party
Independent
Independent Nationalist Front
Indian National Congress
Indian National League
Indian National Socialistic Action Forces
Indian Union Muslim League
Indian Unity Centre
Jamat-E-Seratul Mustakim
Jan Sangh Party
Jatiya Unnayan Party
Jh

In [2]:
"""
Collapse the wide party-level CSV into 3 alliance buckets:

    AITC+   = AITC + Bharatiya Gorkha Prajatantrik Morcha
    BJP     = BJP (alone)
    Other   = everything else (Left Front+, INC, ISF, small parties, Independents, NOTA)

Per constituency, for each bucket:
    <bucket>_total_votes  = sum of votes across member parties
    <bucket>_vote_share   = sum of vote share across member parties (in %)
    <bucket>_candidate    = candidate name of the bucket's highest-voted party

Source: official seat-sharing lists for the 2026 WB Assembly election:
    AITC+ -> AITC 291 + BGPM 3
    BJP   -> 294 (no allies)
    LF+, INC, SUCI(C), BSP, AJUP, AIMIM, etc. all contested separately -> "Other"
"""

import pandas as pd

INPUT_CSV  = "wb_2026_results.csv"
OUTPUT_CSV = "wb_2026_grouped.csv"

AITC_PLUS = {
    "All India Trinamool Congress",
    "Bharatiya Gorkha Prajatantrik Morcha",
}
BJP_ONLY = {
    "Bharatiya Janata Party",
}
GROUPS = ["AITC+", "BJP", "Other"]


def group_of(party: str) -> str:
    if party in AITC_PLUS:
        return "AITC+"
    if party in BJP_ONLY:
        return "BJP"
    return "Other"


def main():
    df = pd.read_csv(INPUT_CSV)

    # discover every party present in the file
    parties = sorted({
        c.removesuffix("_total_votes")
        for c in df.columns
        if c.endswith("_total_votes")
    })

    out = pd.DataFrame({"constituency": df["constituency"]})

    for grp in GROUPS:
        grp_parties = [p for p in parties if group_of(p) == grp]
        if not grp_parties:
            out[f"{grp}_candidate"]   = ""
            out[f"{grp}_total_votes"] = 0
            out[f"{grp}_vote_share"]  = 0.0
            continue

        votes_cols = [f"{p}_total_votes" for p in grp_parties]
        share_cols = [f"{p}_vote_share"  for p in grp_parties]

        votes = df[votes_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
        share = df[share_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

        # bucket totals
        out[f"{grp}_total_votes"] = votes.sum(axis=1).astype(int)
        out[f"{grp}_vote_share"]  = share.sum(axis=1).round(2)

        # candidate = highest-voted party's candidate, per row
        top_idx = votes.values.argmax(axis=1)
        top_party_per_row = [grp_parties[i] for i in top_idx]
        cand = []
        for row_i, p in enumerate(top_party_per_row):
            v = votes.iat[row_i, top_idx[row_i]]
            if v == 0:                                   # nobody from this group contested
                cand.append("")
            else:
                name = df.at[row_i, f"{p}_candidate"]
                cand.append("" if pd.isna(name) else name)
        out[f"{grp}_candidate"] = cand

    # arrange columns: constituency, then candidate/votes/share for each bucket
    col_order = ["constituency"]
    for grp in GROUPS:
        col_order += [f"{grp}_candidate", f"{grp}_total_votes", f"{grp}_vote_share"]
    out = out[col_order]

    out.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote {len(out)} rows -> {OUTPUT_CSV}")
    print(out.head().to_string(index=False))


if __name__ == "__main__":
    main()

Wrote 294 rows -> wb_2026_grouped.csv
         constituency         AITC+_candidate  AITC+_total_votes  AITC+_vote_share  BJP_candidate  BJP_total_votes  BJP_vote_share     Other_candidate  Other_total_votes  Other_vote_share
     ALIPURDUARS - 12          SUMAN KANJILAL              72822             31.33   PARITOSH DAS           143242           61.62         SHYAMAL ROY              14948              6.44
        AMDANGA - 102 MOHAMMAD KASEM SIDDIQUE              81670             38.25    ARINDAM DEY            78675           36.85      BISWAJIT MAITY              52367             24.52
           AMTA - 181      SUKANTA KUMAR PAUL             100195             42.28   AMIT SAMANTA           104649           44.16 JOSHIMUDDIN MALLICK              29696             12.53
        ARAMBAG - 200                BAG MITA              94041             39.87    BAG HEMANTA           123000           52.15      BITHIKA PANDIT              18834              7.98
ASANSOL DAKSHIN - 280 

In [3]:
df = pd.read_csv("wb_2026_grouped.csv")
df["constituency"] = df["constituency"].str.replace(r"\s*-\s*\d+\s*$", "", regex=True)
df.to_csv("wb_2026_grouped.csv", index=False)

df.head()

,constituency,AITC+_candidate,AITC+_total_votes,AITC+_vote_share,BJP_candidate,BJP_total_votes,BJP_vote_share,Other_candidate,Other_total_votes,Other_vote_share
0,ALIPURDUARS,SUMAN KANJILAL,72822,31.33,PARITOSH DAS,143242,61.62,SHYAMAL ROY,14948,6.44
1,AMDANGA,MOHAMMAD KASEM SIDDIQUE,81670,38.25,ARINDAM DEY,78675,36.85,BISWAJIT MAITY,52367,24.52
2,AMTA,SUKANTA KUMAR PAUL,100195,42.28,AMIT SAMANTA,104649,44.16,JOSHIMUDDIN MALLICK,29696,12.53
3,ARAMBAG,BAG MITA,94041,39.87,BAG HEMANTA,123000,52.15,BITHIKA PANDIT,18834,7.98
4,ASANSOL DAKSHIN,TAPAS BANERJEE,78743,36.49,AGNIMITRA PAUL,119582,55.42,SILPI CHAKRABORTY,16678,7.71


In [9]:
grouped["constituency"] = grouped["constituency"].str.strip().str.title()
grouped.to_csv("wb_2026_grouped.csv", index=False)


In [ ]:
# pip install rapidfuzz

In [10]:
import pandas as pd
from rapidfuzz import process, fuzz

grouped = pd.read_csv("wb_2026_grouped.csv")
wb_wide = pd.read_csv("../vglmer_projections/WB_wide.csv")

def normalize(s):
    return s.strip().lower().replace("-", " ").replace(".", "")

a_raw = grouped["constituency"].dropna().str.strip()
b_raw = wb_wide["constituency_name"].dropna().str.strip()

a = set(a_raw)
b = set(b_raw)

a_norm = {normalize(x): x for x in a}
b_norm = {normalize(x): x for x in b}

common_exact = a & b
only_eci = a - b
only_wide = b - a

print(f"grouped only           : {len(a)}")
print(f"WB_wide only           : {len(b)}")
print(f"common (exact)         : {len(common_exact)}")
print(f"in grouped, not in wide: {len(only_eci)}")
print(f"in wide, not in grouped: {len(only_wide)}")

# Fuzzy match: for each unmatched in grouped, find closest in wide
THRESHOLD = 80
print("\n--- fuzzy matches (likely same, written differently) ---")
fuzzy_pairs = []
for name in sorted(only_eci):
    norm = normalize(name)
    match, score, _ = process.extractOne(norm, list(b_norm.keys()), scorer=fuzz.token_sort_ratio)
    if score >= THRESHOLD:
        fuzzy_pairs.append((name, b_norm[match], score))
        print(f"  {name!r:40s} <-> {b_norm[match]!r:40s}  (score={score})")

print(f"\n--- truly only in grouped ({len(only_eci) - len(fuzzy_pairs)}) ---")
matched_eci = {p[0] for p in fuzzy_pairs}
for x in sorted(only_eci - matched_eci):
    print(" ", x)

matched_wide = {p[1] for p in fuzzy_pairs}
print(f"\n--- truly only in WB_wide ({len(only_wide) - len(matched_wide)}) ---")
for x in sorted(only_wide - matched_wide):
    print(" ", x)


grouped only           : 293
WB_wide only           : 294
common (exact)         : 272
in grouped, not in wide: 21
in wide, not in grouped: 22

--- fuzzy matches (likely same, written differently) ---
  'Balarampur'                             <-> 'Baharampur'                              (score=90.0)
  'Bardhaman Dakshin'                      <-> 'Burdwan Dakshin'                         (score=81.25)
  'Bhagawangola'                           <-> 'Bhagabangola'                            (score=91.66666666666666)
  'Bishnupur'                              <-> 'Binpur'                                  (score=80.0)
  'Chowrangee'                             <-> 'Chowranghee'                             (score=95.23809523809523)
  'Dabgram-Fulbari'                        <-> 'Dabgram-Phulbari'                        (score=90.32258064516128)
  'Harischandrapur'                        <-> 'Harishchandrapur'                        (score=96.7741935483871)
  'Indus'                        

In [13]:
grouped = pd.read_csv("wb_2026_grouped.csv")
rename_map = {
    'Bardhaman Dakshin':     'Burdwan Dakshin',
    'Bardhaman Uttar':       'Burdwan Uttar',
    'Bhagawangola':          'Bhagabangola',
    'Chowrangee':            'Chowranghee',
    'Coochbehar Dakshin':    'Cooch Behar Dakshin',
    'Coochbehar Uttar':      'Cooch Behar Uttar',
    'Dabgram-Fulbari':       'Dabgram-Phulbari',
    'Harischandrapur':       'Harishchandrapur',
    'Indus':                 'Indas',
    'Joypur':                'Joypur (Purulia)',
    'Kashipur-Belgachhia':   'Kashipur Belgachhia',
    'Labpur':                'Labhpur',
    'Maniktala':             'Maniktola',
    'Monteswar':             'Manteswar',
    'Nowda':                 'Naoda',
    'Pandabeswar':           'Pandaveswar',
    'Balarampur':            'Balarampur (Purulia)',
    'Raghunathpur':          'Raghunathpur (Purulia)',
    'Raipur':                'Raipur (Bankura)',
    'Ramnagar':              'Ramnagar (Purba Medinipur)',
}

grouped["constituency"] = grouped["constituency"].replace(rename_map)

grouped.loc[grouped["AITC+_candidate"] == "TANMAY GHOSH (BUMBA)", "constituency"] = "Bishnupur (Bankura)"
grouped.loc[grouped["AITC+_candidate"] == "DILIP MONDAL", "constituency"] = "Bishnupur (South 24 Parganas)"

grouped.to_csv("wb_2026_grouped.csv", index=False)


In [14]:
grouped = pd.read_csv("wb_2026_grouped.csv")
wb_wide = pd.read_csv("../vglmer_projections/WB_wide.csv")

a = set(grouped["constituency"].dropna().str.strip())
b = set(wb_wide["constituency_name"].dropna().str.strip())

common = a & b
print(f"grouped: {len(a)}  |  wb_wide: {len(b)}  |  common: {len(common)}")

print("\n--- only in grouped ---")
for x in sorted(a - b): print(" ", x)

print("\n--- only in wb_wide ---")
for x in sorted(b - a): print(" ", x)

grouped: 294  |  wb_wide: 294  |  common: 294

--- only in grouped ---

--- only in wb_wide ---


In [16]:
grouped.head()

,constituency,AITC+_candidate,AITC+_total_votes,AITC+_vote_share,BJP_candidate,BJP_total_votes,BJP_vote_share,Other_candidate,Other_total_votes,Other_vote_share
0,Alipurduars,SUMAN KANJILAL,72822,31.33,PARITOSH DAS,143242,61.62,SHYAMAL ROY,14948,6.44
1,Amdanga,MOHAMMAD KASEM SIDDIQUE,81670,38.25,ARINDAM DEY,78675,36.85,BISWAJIT MAITY,52367,24.52
2,Amta,SUKANTA KUMAR PAUL,100195,42.28,AMIT SAMANTA,104649,44.16,JOSHIMUDDIN MALLICK,29696,12.53
3,Arambag,BAG MITA,94041,39.87,BAG HEMANTA,123000,52.15,BITHIKA PANDIT,18834,7.98
4,Asansol Dakshin,TAPAS BANERJEE,78743,36.49,AGNIMITRA PAUL,119582,55.42,SILPI CHAKRABORTY,16678,7.71


In [17]:
wb_wide.head()

,constituency_name,median_AITC,median_NDA,median_OTHER,ci_low_AITC,ci_low_NDA,ci_low_OTHER,ci_high_AITC,ci_high_NDA,ci_high_OTHER,winner,AITC_2021,NDA_2021,OTHER_2021,aitc_swing,nda_swing,other_swing,margin,surveyed
0,Alipurduars,39.926378,45.423630,14.649993,31.976045,36.451295,11.650583,32.963239,37.313244,12.166182,NDA,0.410,0.482,0.108,-1.073622,-2.776370,3.849993,-5.497252,True
1,Amdanga,40.369518,36.522638,23.107844,32.174586,29.027380,18.241371,33.110645,30.004189,19.113608,AITC,0.420,0.300,0.280,-1.630482,6.522638,-4.892156,3.846880,True
2,Amta,43.562931,40.207836,16.229233,34.756634,32.090506,12.872777,35.584746,32.790494,13.345275,AITC,0.491,0.365,0.144,-5.537069,3.707836,1.829233,3.355094,True
3,Arambag,41.088645,44.900297,14.011057,32.907999,36.042419,11.140383,33.791778,36.810605,11.623887,NDA,0.436,0.469,0.095,-2.511355,-1.999703,4.511057,-3.811652,True
4,Asansol Dakshin,40.735759,44.102369,15.161872,32.194648,34.926252,11.926723,33.116963,35.714136,12.393943,NDA,0.428,0.451,0.121,-2.064241,-0.997631,3.061872,-3.366611,True
